# Notebook 6 — Baseline Improvement: Feature Engineering + Optuna + Stacking
### XGBoost advancement on natural class distribution (no SMOTE)

**Baseline:** Default XGBoost, 57 raw features, no tuning → **Accuracy ≈ 0.9577**  
**Goal:** Exceed baseline purely through algorithmic improvements — no data resampling or row/feature cleaning  
*(Those are covered separately in Notebook 12)*

**Improvement pipeline:**
1. Feature engineering — log1p transform + 7 interaction features (64 total)
2. Optuna Bayesian hyperparameter optimisation (XGBoost 150 trials, RF 80 trials)
3. Stacking ensemble (XGBoost + RF + SVM + GB → meta: LR)
4. Permutation-selected feature subset
5. Friedman + Nemenyi post-hoc statistical test
6. Robustness study (10 random seeds)
7. Cost-sensitive learning
8. Calibrated best model
9. Consolidated results table

In [106]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import copy, os, time

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    cross_val_predict, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier, VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
    average_precision_score, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# scikit-posthocs for Nemenyi test
try:
    import scikit_posthocs as sp
    HAS_SP = True
except ImportError:
    HAS_SP = False
    print('scikit-posthocs not installed — Nemenyi will be skipped')

from scipy import stats

os.makedirs('advanced_figures', exist_ok=True)

SEED = 0
np.random.seed(SEED)
print('Libraries loaded OK')

Libraries loaded OK


## 1. Data Loading & Best Pipeline

In [107]:
col_names = (
    [f'word_freq_{w}' for w in [
        'make','address','all','3d','our','over','remove','internet','order','mail',
        'receive','will','people','report','addresses','free','business','email','you',
        'credit','your','font','000','money','hp','hpl','george','650','lab','labs',
        'telnet','857','data','415','85','technology','1999','parts','pm','direct',
        'cs','meeting','original','project','re','edu','table','conference']] +
    [f'char_freq_{c}' for c in ['semicolon','lparen','lbracket','exclaim','dollar','hash']] +
    ['capital_run_length_average','capital_run_length_longest',
     'capital_run_length_total','spam']
)

# ── Load the same dataset as Stage 4 feature engineering (preprocessed_data_v2.csv)
# Filter to original 57 features only, matching notebook 05
df_raw = pd.read_csv('preprocessed_data_v2.csv')
ENG_PREFIXES = [
    'spam_word', 'ham_word', 'spam_ham', 'char_spam',
    'char_ham', 'capital_ratio', 'word_present', 'word_diversity'
]
ORIG_COLS = [c for c in df_raw.columns
             if c != 'spam' and not any(c.startswith(p) for p in ENG_PREFIXES)]

X_raw = df_raw[ORIG_COLS].values.astype(np.float64)
y     = df_raw['spam'].values

print(f'Dataset (preprocessed_data_v2.csv, 57 orig feats): {X_raw.shape[0]} rows, {X_raw.shape[1]} features | spam={y.mean():.1%}')

WORD_FEATS = [c for c in ORIG_COLS if c.startswith('word_freq_')]
CHAR_FEATS = [c for c in ORIG_COLS if c.startswith('char_freq_')]
CAP_FEATS  = [c for c in ORIG_COLS if c.startswith('capital_run_')]
FREQ_IDX   = [ORIG_COLS.index(c) for c in WORD_FEATS + CHAR_FEATS]
ALL_IDX    = list(range(len(ORIG_COLS)))

# ── Log1p transform on word/char frequency features ────────────────────────
def log1p_freq(X):
    Xc = X.copy()
    Xc[:, FREQ_IDX] = np.log1p(Xc[:, FREQ_IDX])
    return Xc

# ── 7 interaction features derived from error analysis ─────────────────────
def add_interactions(X, orig_cols):
    df_tmp = pd.DataFrame(X, columns=orig_cols)
    get = lambda n: df_tmp.get(n, pd.Series(np.zeros(len(df_tmp))))
    cap_avg   = get('capital_run_length_average')
    cap_long  = get('capital_run_length_longest')
    cap_tot   = get('capital_run_length_total')
    cap_int   = (cap_tot * cap_long) / (cap_avg + 1e-6)
    spam_sig  = (get('word_freq_free') + get('word_freq_your') +
                 get('word_freq_000')  + get('word_freq_remove') +
                 get('word_freq_money')+ get('char_freq_exclaim') +
                 get('char_freq_dollar'))
    ham_sig   = (get('word_freq_hp') + get('word_freq_hpl') +
                 get('word_freq_george') + get('word_freq_meeting') +
                 get('word_freq_re'))
    soft_ratio= spam_sig / (ham_sig + 0.01)
    cap_x_exc = np.log1p(cap_tot) * np.log1p(get('char_freq_exclaim'))
    cap_x_dol = np.log1p(cap_tot) * np.log1p(get('char_freq_dollar'))
    word_div  = (df_tmp[WORD_FEATS] > 0).sum(axis=1)
    extras = np.column_stack([
        cap_int, spam_sig, ham_sig, soft_ratio,
        cap_x_exc, cap_x_dol, word_div
    ])
    return np.hstack([X, extras])

# ── Build engineered dataset ────────────────────────────────────────────────
X_log = log1p_freq(X_raw)
X_eng = add_interactions(X_log, ORIG_COLS)   # 57 + 7 = 64 features

# ── Stratified 80/20 split ──────────────────────────────────────────────────
X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=SEED, stratify=y)
X_tr_eng, X_te_eng, _, _ = train_test_split(
    X_eng, y, test_size=0.20, random_state=SEED, stratify=y)

# ── Fit StandardScaler on training set only ────────────────────────────────
scaler_raw = StandardScaler().fit(X_tr_raw)
scaler_eng = StandardScaler().fit(X_tr_eng)

X_train_s  = scaler_raw.transform(X_tr_raw)
X_test_s   = scaler_raw.transform(X_te_raw)
X_train_es = scaler_eng.transform(X_tr_eng)
X_test_es  = scaler_eng.transform(X_te_eng)

print(f'Train: {X_train_s.shape} | spam={y_train.mean():.1%}')
print(f'Test : {X_test_s.shape}  | spam={y_test.mean():.1%}')
print(f'Engineered train/test: {X_train_es.shape}, {X_test_es.shape}')

# ── Baseline: default XGBoost, 57 raw features, no tuning ─────────────────
xgb_default = XGBClassifier(eval_metric='logloss', random_state=SEED,
                             n_jobs=-1, verbosity=0)
xgb_default.fit(X_train_s, y_train)
base_acc = accuracy_score(y_test, xgb_default.predict(X_test_s))
print(f'\nBaseline (default XGBoost, 57 feat, no tuning) → Acc = {base_acc:.4f}')

Dataset (preprocessed_data_v2.csv, 57 orig feats): 5062 rows, 57 features | spam=50.0%
Train: (4049, 57) | spam=50.0%
Test : (1013, 57)  | spam=50.0%
Engineered train/test: (4049, 64), (1013, 64)

Baseline (default XGBoost, 57 feat, no tuning) → Acc = 0.9576


In [108]:
# ── Universal evaluate helper ──────────────────────────────────────────────
def evaluate(name, clf, Xtr, Xte, ytr, yte, verbose=True):
    clf = copy.deepcopy(clf)
    clf.fit(Xtr, ytr)
    yp    = clf.predict(Xte)
    yprob = clf.predict_proba(Xte)[:, 1]
    cm    = confusion_matrix(yte, yp)
    tn, fp, fn, tp = cm.ravel()
    res = {
        'name': name,
        'acc':      accuracy_score(yte, yp),
        'bal_acc':  balanced_accuracy_score(yte, yp),
        'f1':       f1_score(yte, yp),
        'prec':     precision_score(yte, yp),
        'rec':      recall_score(yte, yp),
        'auc':      roc_auc_score(yte, yprob),
        'ap':       average_precision_score(yte, yprob),
        'mcc':      matthews_corrcoef(yte, yp),
        'kappa':    cohen_kappa_score(yte, yp),
        'brier':    brier_score_loss(yte, yprob),
        'logloss':  log_loss(yte, yprob),
        'fp': int(fp), 'fn': int(fn),
        'cm': cm, 'y_pred': yp, 'y_prob': yprob, 'clf': clf,
    }
    if verbose:
        print(f"{name:<35} Acc={res['acc']:.4f}  F1={res['f1']:.4f}  "
              f"AUC={res['auc']:.4f}  MCC={res['mcc']:.4f}  "
              f"FP={res['fp']:3d}  FN={res['fn']:3d}")
    return res

print('evaluate() defined')

evaluate() defined


## 2. Optuna Bayesian Hyperparameter Optimisation
Bayesian TPE search is far more sample-efficient than grid/random search.  
We run 150 trials for XGBoost and 80 trials for RandomForest on the engineered feature set.

In [109]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Natural class ratio — used as reference point for scale_pos_weight search
_neg = (y_train == 0).sum()
_pos = (y_train == 1).sum()
_natural_spw = _neg / _pos
print(f'Natural scale_pos_weight (neg/pos): {_natural_spw:.3f}')

# ── XGBoost Optuna study ───────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 3, 9),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        # ── class-imbalance compensation ─────────────────────────────────────
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight', 0.5, _natural_spw * 2),
        'eval_metric': 'logloss', 'use_label_encoder': False,
        'random_state': SEED, 'n_jobs': -1
    }
    clf = XGBClassifier(**params)
    scores = cross_val_score(clf, X_train_es, y_train, cv=cv5,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

print('Running Optuna XGBoost study (150 trials)...')
t0 = time.time()
xgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=150, show_progress_bar=False)
print(f'Done in {time.time()-t0:.0f}s | best AUC={xgb_study.best_value:.4f}')
print('Best XGB params:', xgb_study.best_params)

Natural scale_pos_weight (neg/pos): 1.000
Running Optuna XGBoost study (150 trials)...
Done in 93s | best AUC=0.9897
Best XGB params: {'n_estimators': 548, 'max_depth': 6, 'learning_rate': 0.029292620700111325, 'subsample': 0.975737447954019, 'colsample_bytree': 0.5777408318849939, 'min_child_weight': 1, 'reg_alpha': 0.029884395318706788, 'reg_lambda': 0.18063879297683233, 'gamma': 0.1684603880212301, 'scale_pos_weight': 1.2760166763345995}


In [110]:
# ── RF Optuna study ───────────────────────────────────────────────────────
def rf_objective(trial):
    params = {
        'n_estimators':   trial.suggest_int('n_estimators', 100, 600),
        'max_depth':      trial.suggest_int('max_depth', 5, 30),
        'max_features':   trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5]),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_samples':    trial.suggest_float('max_samples', 0.5, 1.0),
        'random_state': SEED, 'n_jobs': -1
    }
    clf = RandomForestClassifier(**params)
    scores = cross_val_score(clf, X_train_es, y_train, cv=cv5,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

print('Running Optuna RF study (80 trials)...')
t0 = time.time()
rf_study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
rf_study.optimize(rf_objective, n_trials=80, show_progress_bar=False)
print(f'Done in {time.time()-t0:.0f}s | best AUC={rf_study.best_value:.4f}')
print('Best RF params:', rf_study.best_params)

Running Optuna RF study (80 trials)...


/Users/sithijaseneviratne/Documents/uni work/ml_paper/venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/sithijaseneviratne/Documents/uni work/ml_paper/venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/sithijaseneviratne/Documents/uni work/ml_paper/venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to th

Done in 130s | best AUC=0.9875
Best RF params: {'n_estimators': 532, 'max_depth': 22, 'max_features': 'log2', 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_samples': 0.8398801907716124}


In [111]:
# ── Evaluate Optuna-tuned models ──────────────────────────────────────────
xgb_best_params = {**xgb_study.best_params,
                   'eval_metric': 'logloss', 'random_state': SEED, 'n_jobs': -1}
xgb_opt = XGBClassifier(**xgb_best_params)

rf_best_params = {**rf_study.best_params, 'random_state': SEED, 'n_jobs': -1}
rf_opt = RandomForestClassifier(**rf_best_params)

print('=== Optuna-Tuned Models (Engineered Features) ===')
res_xgb_opt = evaluate('XGBoost (Optuna)', xgb_opt, X_train_es, X_test_es, y_train, y_test)
res_rf_opt  = evaluate('RF (Optuna)',       rf_opt,  X_train_es, X_test_es, y_train, y_test)

# Baseline for comparison (engineered, default params)
xgb_base = XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1)
rf_base  = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
print('\n=== Baseline (Default Params, Engineered Features) ===')
res_xgb_base = evaluate('XGBoost (default)', xgb_base, X_train_es, X_test_es, y_train, y_test)
res_rf_base  = evaluate('RF (default)',       rf_base,  X_train_es, X_test_es, y_train, y_test)

=== Optuna-Tuned Models (Engineered Features) ===
XGBoost (Optuna)                    Acc=0.9625  F1=0.9627  AUC=0.9909  MCC=0.9250  FP= 22  FN= 16
RF (Optuna)                         Acc=0.9576  F1=0.9573  AUC=0.9880  MCC=0.9151  FP= 19  FN= 24

=== Baseline (Default Params, Engineered Features) ===
XGBoost (default)                   Acc=0.9595  F1=0.9597  AUC=0.9910  MCC=0.9191  FP= 23  FN= 18
RF (default)                        Acc=0.9536  F1=0.9534  AUC=0.9877  MCC=0.9072  FP= 22  FN= 25


In [112]:
# ── Optuna optimisation history plot ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, study, label in zip(axes,
    [xgb_study, rf_study], ['XGBoost', 'Random Forest']):
    trials_df = study.trials_dataframe()
    ax.plot(trials_df['number'], trials_df['value'], '.', alpha=0.4, ms=4)
    best_so_far = trials_df['value'].cummax()
    ax.plot(trials_df['number'], best_so_far, 'r-', lw=2, label='Best so far')
    ax.set_xlabel('Trial'); ax.set_ylabel('CV AUC (5-fold)')
    ax.set_title(f'{label} — Optuna Optimisation History')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('advanced_figures/optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/optuna_history.png')

Saved: advanced_figures/optuna_history.png


## 3. Stacking Ensemble
Base learners: Optuna-XGBoost, Optuna-RF, SVM, Gradient Boosting  
Meta-learner: Logistic Regression (uses out-of-fold predictions → no leakage)

In [113]:
# ── Define base learners (re-fit will happen inside StackingClassifier) ───
base_estimators = [
    ('xgb', XGBClassifier(**xgb_best_params)),
    ('rf',  RandomForestClassifier(**rf_best_params)),
    ('svm', SVC(C=10, kernel='rbf', probability=True, random_state=SEED)),
    ('gb',  GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                        learning_rate=0.05, random_state=SEED)),
]
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)

stacker = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_lr,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)

print('Training stacking ensemble...')
t0 = time.time()
res_stack = evaluate('Stacking (XGB+RF+SVM+GB → LR)',
                     stacker, X_train_es, X_test_es, y_train, y_test)
print(f'  Elapsed: {time.time()-t0:.0f}s')

# Also try soft voting for comparison
voter = VotingClassifier(
    estimators=base_estimators,
    voting='soft', n_jobs=-1
)
print('Training soft-voting ensemble...')
t0 = time.time()
res_vote = evaluate('Soft Voting (XGB+RF+SVM+GB)',
                    voter, X_train_es, X_test_es, y_train, y_test)
print(f'  Elapsed: {time.time()-t0:.0f}s')

Training stacking ensemble...
Stacking (XGB+RF+SVM+GB → LR)       Acc=0.9635  F1=0.9634  AUC=0.9903  MCC=0.9270  FP= 18  FN= 19
  Elapsed: 7s
Training soft-voting ensemble...
Soft Voting (XGB+RF+SVM+GB)         Acc=0.9605  F1=0.9605  AUC=0.9897  MCC=0.9210  FP= 20  FN= 20
  Elapsed: 3s


## 4. Permutation-Selected Feature Subset
Drop features with near-zero cross-validated permutation importance, then re-train.

In [114]:
# ── Cross-validated permutation importance on engineered set ──────────────
print('Computing CV permutation importance (5 folds × 10 repeats)...')
N_FEATS = X_train_es.shape[1]
feat_names_eng = ORIG_COLS + [
    'cap_intensity','spam_signal','ham_signal','spam_ham_soft',
    'cap_x_exclaim','cap_x_dollar','word_diversity'
]

xgb_perm = XGBClassifier(**xgb_best_params)
fold_imps = []
for tr_idx, va_idx in cv5.split(X_train_es, y_train):
    Xf_tr, yf_tr = X_train_es[tr_idx], y_train[tr_idx]
    Xf_va, yf_va = X_train_es[va_idx], y_train[va_idx]
    clf_tmp = copy.deepcopy(xgb_perm)
    clf_tmp.fit(Xf_tr, yf_tr)
    pi = permutation_importance(clf_tmp, Xf_va, yf_va,
                                n_repeats=10, random_state=SEED, n_jobs=-1)
    fold_imps.append(pi.importances_mean)

perm_mean = np.mean(fold_imps, axis=0)
perm_std  = np.std(fold_imps, axis=0)

# rank by mean
rank_idx = np.argsort(perm_mean)[::-1]
print(f'Top-15 features by CV permutation importance:')
for i in rank_idx[:15]:
    print(f'  {feat_names_eng[i]:<40} {perm_mean[i]:.4f} ± {perm_std[i]:.4f}')

Computing CV permutation importance (5 folds × 10 repeats)...
Top-15 features by CV permutation importance:
  word_freq_edu                            0.0114 ± 0.0022
  ham_signal                               0.0086 ± 0.0045
  word_freq_our                            0.0085 ± 0.0028
  spam_ham_soft                            0.0078 ± 0.0041
  word_freq_remove                         0.0075 ± 0.0010
  word_freq_george                         0.0060 ± 0.0013
  spam_signal                              0.0060 ± 0.0034
  capital_run_length_average               0.0052 ± 0.0022
  cap_x_exclaim                            0.0049 ± 0.0023
  word_freq_650                            0.0041 ± 0.0022
  capital_run_length_longest               0.0040 ± 0.0019
  cap_intensity                            0.0034 ± 0.0014
  capital_run_length_total                 0.0034 ± 0.0016
  word_freq_credit                         0.0032 ± 0.0021
  word_freq_business                       0.0027 ± 0.0018


In [115]:
# ── Feature importance bar chart ──────────────────────────────────────────
TOP_N = 20
fig, ax = plt.subplots(figsize=(10, 7))
top_idx = rank_idx[:TOP_N]
bars = ax.barh(range(TOP_N), perm_mean[top_idx][::-1],
               xerr=perm_std[top_idx][::-1], capsize=3,
               color=plt.cm.viridis(np.linspace(0.2, 0.8, TOP_N)))
ax.set_yticks(range(TOP_N))
ax.set_yticklabels([feat_names_eng[i] for i in top_idx[::-1]], fontsize=9)
ax.set_xlabel('Mean decrease in accuracy (permutation importance)')
ax.set_title(f'Top-{TOP_N} Features — CV Permutation Importance (XGBoost-Optuna)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('advanced_figures/perm_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/perm_importance.png')

Saved: advanced_figures/perm_importance.png


In [116]:
# ── Select top-K features and re-evaluate ────────────────────────────────
# threshold: keep features where mean > 0
keep_mask_pos = perm_mean > 0
keep_mask_top30 = np.zeros(N_FEATS, dtype=bool)
keep_mask_top30[rank_idx[:30]] = True

print(f'Features with positive permutation importance: {keep_mask_pos.sum()}')
print(f'Top-30 features selected')

X_tr_pos  = X_train_es[:, keep_mask_pos]
X_te_pos  = X_test_es[:, keep_mask_pos]
X_tr_top30= X_train_es[:, keep_mask_top30]
X_te_top30= X_test_es[:, keep_mask_top30]

print('\n=== Permutation-Selected Feature Subsets (XGBoost-Optuna) ===')
res_pos   = evaluate(f'XGB-Opt, +perm features ({keep_mask_pos.sum()})',
                     XGBClassifier(**xgb_best_params),
                     X_tr_pos, X_te_pos, y_train, y_test)
res_top30 = evaluate('XGB-Opt, top-30 features',
                     XGBClassifier(**xgb_best_params),
                     X_tr_top30, X_te_top30, y_train, y_test)

Features with positive permutation importance: 54
Top-30 features selected

=== Permutation-Selected Feature Subsets (XGBoost-Optuna) ===
XGB-Opt, +perm features (54)        Acc=0.9645  F1=0.9647  AUC=0.9909  MCC=0.9290  FP= 22  FN= 14
XGB-Opt, top-30 features            Acc=0.9664  F1=0.9667  AUC=0.9900  MCC=0.9331  FP= 22  FN= 12


## 5. Friedman + Nemenyi Post-Hoc Statistical Test
Gold standard for comparing multiple classifiers across k folds.
Uses 10-fold CV accuracy scores for each model, then Friedman chi-square followed by Nemenyi pairwise test.

In [117]:
# ── Collect 10-fold CV scores for key models ──────────────────────────────
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

stat_models = {
    'XGB-Default':  XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1),
    'XGB-Optuna':   XGBClassifier(**xgb_best_params),
    'RF-Default':   RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    'RF-Optuna':    RandomForestClassifier(**rf_best_params),
    'SVM':          SVC(C=10, kernel='rbf', probability=True, random_state=SEED),
    'GB':           GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                               learning_rate=0.05, random_state=SEED),
    'LogReg':       LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    'KNN':          KNeighborsClassifier(n_neighbors=5),
}

print('Computing 10-fold CV accuracy for each model (engineered features)...')
cv_scores = {}
for name, clf in stat_models.items():
    sc = cross_val_score(clf, X_train_es, y_train,
                         cv=cv10, scoring='accuracy', n_jobs=-1)
    cv_scores[name] = sc
    print(f'  {name:<20} {sc.mean():.4f} ± {sc.std():.4f}')

Computing 10-fold CV accuracy for each model (engineered features)...
  XGB-Default          0.9553 ± 0.0108
  XGB-Optuna           0.9595 ± 0.0081
  RF-Default           0.9467 ± 0.0082
  RF-Optuna            0.9467 ± 0.0090
  SVM                  0.9509 ± 0.0086
  GB                   0.9481 ± 0.0102
  LogReg               0.9390 ± 0.0082
  KNN                  0.9296 ± 0.0087


In [118]:
# ── Friedman test ─────────────────────────────────────────────────────────
score_matrix = np.array([cv_scores[k] for k in cv_scores])  # (n_models, n_folds)
stat, p_friedman = stats.friedmanchisquare(*[cv_scores[k] for k in cv_scores])
print(f'Friedman test: chi2={stat:.4f}, p={p_friedman:.6f}')
if p_friedman < 0.05:
    print('Result: Significant differences exist (p < 0.05) → proceed to Nemenyi post-hoc')
else:
    print('Result: No significant differences found')

# ── Nemenyi post-hoc ──────────────────────────────────────────────────────
if HAS_SP and p_friedman < 0.05:
    nemenyi_df = pd.DataFrame(cv_scores).T  # shape: (n_models, n_folds)
    nemenyi_p = sp.posthoc_nemenyi_friedman(nemenyi_df.T.values)
    nemenyi_p.columns = list(cv_scores.keys())
    nemenyi_p.index   = list(cv_scores.keys())
    print('\nNemenyi pairwise p-values:')
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(nemenyi_p)
elif not HAS_SP:
    # Manual Nemenyi using critical difference
    print('\nskikit-posthocs not available — manual critical differences shown instead')
    n_models = len(cv_scores)
    n_folds  = 10
    # Wilcoxon signed-rank as proxy pairwise tests with Bonferroni
    names = list(cv_scores.keys())
    alpha = 0.05 / (n_models*(n_models-1)//2)
    print(f'Bonferroni-corrected alpha: {alpha:.5f}')
    sig_pairs = []
    for i in range(n_models):
        for j in range(i+1, n_models):
            w, p = stats.wilcoxon(cv_scores[names[i]], cv_scores[names[j]])
            if p < alpha:
                sig_pairs.append((names[i], names[j], p))
    print(f'\nSignificant pairs (Wilcoxon + Bonferroni) [{len(sig_pairs)}]:')
    for a,b,p in sig_pairs:
        print(f'  {a} vs {b}: p={p:.6f}')

Friedman test: chi2=49.5685, p=0.000000
Result: Significant differences exist (p < 0.05) → proceed to Nemenyi post-hoc

Nemenyi pairwise p-values:
             XGB-Default  XGB-Optuna  RF-Default  RF-Optuna    SVM     GB  \
XGB-Default       1.0000      0.9800      0.3297     0.3297 0.9072 0.5390   
XGB-Optuna        0.9800      1.0000      0.0304     0.0304 0.3297 0.0776   
RF-Default        0.3297      0.0304      1.0000     1.0000 0.9800 1.0000   
RF-Optuna         0.3297      0.0304      1.0000     1.0000 0.9800 1.0000   
SVM               0.9072      0.3297      0.9800     0.9800 1.0000 0.9983   
GB                0.5390      0.0776      1.0000     1.0000 0.9983 1.0000   
LogReg            0.0022      0.0000      0.6947     0.6947 0.1392 0.4761   
KNN               0.0001      0.0000      0.2107     0.2107 0.0143 0.0989   

             LogReg    KNN  
XGB-Default  0.0022 0.0001  
XGB-Optuna   0.0000 0.0000  
RF-Default   0.6947 0.2107  
RF-Optuna    0.6947 0.2107  
SVM          0

In [119]:
# ── Nemenyi heatmap / CV boxplot ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# left: boxplot of 10-fold CV
ax = axes[0]
df_cv = pd.DataFrame(cv_scores)
bp = df_cv.boxplot(ax=ax, rot=30, grid=False, patch_artist=True)
ax.set_ylabel('10-fold CV Accuracy')
ax.set_title('10-fold CV Accuracy Distribution')
ax.set_ylim(0.90, 1.00)
ax.axhline(df_cv.values.max(), color='red', ls='--', lw=1, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

# right: Nemenyi heatmap or Wilcoxon p-value heatmap
ax2 = axes[1]
names_list = list(cv_scores.keys())
n = len(names_list)
p_mat = np.ones((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            _, pv = stats.wilcoxon(cv_scores[names_list[i]],
                                   cv_scores[names_list[j]])
            p_mat[i, j] = pv

sig_mat = (p_mat < 0.05).astype(float)
sns.heatmap(pd.DataFrame(p_mat, index=names_list, columns=names_list),
            ax=ax2, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.1, linewidths=0.5)
ax2.set_title('Pairwise Wilcoxon p-values\n(green=not significant, red=p<0.05)')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha='right', fontsize=8)
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig('advanced_figures/friedman_nemenyi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/friedman_nemenyi.png')

Saved: advanced_figures/friedman_nemenyi.png


## 6. Robustness Study — Multiple Random Seeds
Evaluate best model (XGB-Optuna + engineered features) and RF-Optuna across 10 different random seeds to show variance and confirm results are not artefacts of a single split.

In [120]:
SEEDS = [0, 1, 7, 13, 21, 42, 100, 123, 256, 999]

def eval_seed(seed, clf_class, clf_params, use_eng=True):
    Xsrc = X_eng if use_eng else X_raw
    Xtr, Xte, ytr, yte = train_test_split(
        Xsrc, y, test_size=0.20, random_state=seed, stratify=y)
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
    clf = clf_class(**clf_params)
    clf.fit(Xtr_s, ytr)
    yp    = clf.predict(Xte_s)
    yprob = clf.predict_proba(Xte_s)[:, 1]
    return {
        'acc': accuracy_score(yte, yp),
        'auc': roc_auc_score(yte, yprob),
        'f1':  f1_score(yte, yp),
        'mcc': matthews_corrcoef(yte, yp),
    }

print('Running robustness study across 10 seeds...')
robust_rows = []
for seed in SEEDS:
    r_xgb = eval_seed(seed, XGBClassifier, xgb_best_params)
    r_rf  = eval_seed(seed, RandomForestClassifier, rf_best_params)
    robust_rows.append({'seed': seed, 'model': 'XGB-Optuna', **r_xgb})
    robust_rows.append({'seed': seed, 'model': 'RF-Optuna',  **r_rf})

rob_df = pd.DataFrame(robust_rows)
print(rob_df.groupby('model')[['acc','auc','f1','mcc']].agg(['mean','std']).round(4))

Running robustness study across 10 seeds...
              acc           auc            f1           mcc       
             mean    std   mean    std   mean    std   mean    std
model                                                             
RF-Optuna  0.9520 0.0079 0.9874 0.0028 0.9521 0.0078 0.9041 0.0157
XGB-Optuna 0.9599 0.0065 0.9906 0.0020 0.9602 0.0064 0.9200 0.0130


In [121]:
# ── Robustness boxplots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, metric in zip(axes, ['acc','auc','f1','mcc']):
    data = [rob_df[rob_df['model']==m][metric].values
            for m in ['XGB-Optuna','RF-Optuna']]
    bp = ax.boxplot(data, labels=['XGB-Optuna','RF-Optuna'],
                    patch_artist=True, notch=False)
    bp['boxes'][0].set_facecolor('#2196F3')
    bp['boxes'][1].set_facecolor('#4CAF50')
    ax.set_title(metric.upper()); ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15)

fig.suptitle('Robustness Study — 10 Random Seeds', fontsize=13)
plt.tight_layout()
plt.savefig('advanced_figures/robustness_seeds.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/robustness_seeds.png')

Saved: advanced_figures/robustness_seeds.png


## 7. Cost-Sensitive Learning
In spam detection, false negatives (spam reaching inbox) and false positives (ham going to spam) have different practical costs. We sweep class_weight to show the FP/FN trade-off explicitly.

In [122]:
# ── Vary spam weight (class 1 weight) ────────────────────────────────────
weights = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0]
cost_rows = []
for w in weights:
    cw = {0: 1.0, 1: w}
    xgb_cw = XGBClassifier(**{**xgb_best_params,
                               'scale_pos_weight': w})
    xgb_cw.fit(X_train_es, y_train)
    yp    = xgb_cw.predict(X_test_es)
    yprob = xgb_cw.predict_proba(X_test_es)[:,1]
    cm    = confusion_matrix(y_test, yp)
    tn, fp, fn, tp = cm.ravel()
    n_neg, n_pos = (y_test==0).sum(), (y_test==1).sum()
    cost_rows.append({
        'spam_weight': w,
        'acc': accuracy_score(y_test, yp),
        'f1':  f1_score(y_test, yp),
        'fp': int(fp), 'fn': int(fn),
        'fpr': fp/n_neg, 'fnr': fn/n_pos,
        'mcc': matthews_corrcoef(y_test, yp),
    })
    print(f'  weight={w:.1f}  acc={accuracy_score(y_test,yp):.4f}  '
          f'FP={fp:3d}  FN={fn:3d}  MCC={matthews_corrcoef(y_test,yp):.4f}')

cost_df = pd.DataFrame(cost_rows)

  weight=0.5  acc=0.9645  FP= 17  FN= 19  MCC=0.9289
  weight=1.0  acc=0.9674  FP= 18  FN= 15  MCC=0.9349
  weight=1.5  acc=0.9625  FP= 24  FN= 14  MCC=0.9252
  weight=2.0  acc=0.9605  FP= 26  FN= 14  MCC=0.9213
  weight=3.0  acc=0.9595  FP= 28  FN= 13  MCC=0.9195
  weight=4.0  acc=0.9585  FP= 29  FN= 13  MCC=0.9175
  weight=5.0  acc=0.9556  FP= 33  FN= 12  MCC=0.9119


In [123]:
# ── Cost-sensitivity plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.plot(cost_df['spam_weight'], cost_df['fp'], 'o-', color='blue', label='FP (Ham→Spam)')
ax.plot(cost_df['spam_weight'], cost_df['fn'], 's-', color='red',  label='FN (Spam→Inbox)')
ax.set_xlabel('Spam class weight'); ax.set_ylabel('Error count')
ax.set_title('FP vs FN — Class Weight Trade-off')
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(cost_df['spam_weight'], cost_df['acc'], 'D-', color='green')
ax2.set_xlabel('Spam class weight'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Class Weight')
ax2.grid(alpha=0.3)

ax3 = axes[2]
ax3.plot(cost_df['spam_weight'], cost_df['mcc'], '^-', color='purple')
ax3.set_xlabel('Spam class weight'); ax3.set_ylabel('MCC')
ax3.set_title('MCC vs Class Weight')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('advanced_figures/cost_sensitive.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/cost_sensitive.png')

Saved: advanced_figures/cost_sensitive.png


## 8. Best Model Calibration & Final Evaluation
Calibrate the best model using isotonic regression, then compare raw vs calibrated probability quality.

In [124]:
# ── Calibrate XGB-Optuna ──────────────────────────────────────────────────
xgb_final = XGBClassifier(**xgb_best_params)
xgb_final.fit(X_train_es, y_train)

cal_sigmoid = CalibratedClassifierCV(
    XGBClassifier(**xgb_best_params), method='sigmoid', cv=5)
cal_isotonic = CalibratedClassifierCV(
    XGBClassifier(**xgb_best_params), method='isotonic', cv=5)
cal_sigmoid.fit(X_train_es, y_train)
cal_isotonic.fit(X_train_es, y_train)

from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
for clf, label, color in [
    (xgb_final,    'Uncalibrated XGB', 'steelblue'),
    (cal_sigmoid,  'Sigmoid calibrated','orange'),
    (cal_isotonic, 'Isotonic calibrated','green')
]:
    prob = clf.predict_proba(X_test_es)[:,1]
    frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10)
    ax1.plot(mean_pred, frac_pos, 's-', label=label, color=color)
ax1.plot([0,1],[0,1],'k--',lw=1,label='Perfect')
ax1.set_xlabel('Mean predicted probability')
ax1.set_ylabel('Fraction of positives')
ax1.set_title('Calibration Curves')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2 = axes[1]
metrics_cal = []
for clf, label in [
    (xgb_final,    'Uncalibrated'),
    (cal_sigmoid,  'Sigmoid'),
    (cal_isotonic, 'Isotonic')
]:
    prob = clf.predict_proba(X_test_es)[:,1]
    bs   = brier_score_loss(y_test, prob)
    ll   = log_loss(y_test, prob)
    metrics_cal.append({'Model': label, 'Brier': bs, 'LogLoss': ll})

cal_df = pd.DataFrame(metrics_cal)
x = np.arange(len(cal_df))
w = 0.3
ax2.bar(x - w/2, cal_df['Brier'],   w, label='Brier Score')
ax2.bar(x + w/2, cal_df['LogLoss'], w, label='Log Loss')
ax2.set_xticks(x); ax2.set_xticklabels(cal_df['Model'])
ax2.set_title('Probability Quality (lower = better)')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('advanced_figures/calibration_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCalibration metrics:')
print(cal_df.to_string(index=False))


Calibration metrics:
       Model  Brier  LogLoss
Uncalibrated 0.0292   0.1125
     Sigmoid 0.0303   0.1219
    Isotonic 0.0300   0.1417


## 9. Threshold Optimisation on Best Model
Find the optimal decision threshold for Accuracy, F1, and MCC.

In [125]:
# ── Threshold sweep ───────────────────────────────────────────────────────
xgb_final_fitted = copy.deepcopy(xgb_final)
xgb_final_fitted.fit(X_train_es, y_train)
y_probs_final = xgb_final_fitted.predict_proba(X_test_es)[:, 1]

thresholds = np.linspace(0.20, 0.85, 200)
accs, f1s, mccs = [], [], []
for t in thresholds:
    yp_t = (y_probs_final >= t).astype(int)
    accs.append(accuracy_score(y_test, yp_t))
    f1s.append(f1_score(y_test, yp_t, zero_division=0))
    mccs.append(matthews_corrcoef(y_test, yp_t))

best_t_acc = thresholds[np.argmax(accs)]
best_t_f1  = thresholds[np.argmax(f1s)]
best_t_mcc = thresholds[np.argmax(mccs)]

print(f'Optimal threshold → Acc: {best_t_acc:.3f} ({max(accs):.4f})')
print(f'Optimal threshold → F1:  {best_t_f1:.3f} ({max(f1s):.4f})')
print(f'Optimal threshold → MCC: {best_t_mcc:.3f} ({max(mccs):.4f})')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, accs, label='Accuracy')
ax.plot(thresholds, f1s,  label='F1')
ax.plot(thresholds, mccs, label='MCC')
ax.axvline(best_t_acc, color='C0', ls='--', alpha=0.6)
ax.axvline(best_t_f1,  color='C1', ls='--', alpha=0.6)
ax.axvline(best_t_mcc, color='C2', ls='--', alpha=0.6)
ax.set_xlabel('Decision threshold')
ax.set_title('XGB-Optuna — Threshold Optimisation')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('advanced_figures/threshold_opt.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/threshold_opt.png')

# Evaluate at best accuracy threshold
yp_best_t = (y_probs_final >= best_t_acc).astype(int)
print(f'\nAt threshold={best_t_acc:.3f}:')
print(f'  Acc={accuracy_score(y_test,yp_best_t):.4f}')
print(f'  F1={f1_score(y_test,yp_best_t):.4f}')
print(f'  MCC={matthews_corrcoef(y_test,yp_best_t):.4f}')
cm_t = confusion_matrix(y_test, yp_best_t)
tn,fp,fn,tp = cm_t.ravel()
print(f'  FP={fp}, FN={fn}')

Optimal threshold → Acc: 0.595 (0.9664)
Optimal threshold → F1:  0.595 (0.9663)
Optimal threshold → MCC: 0.683 (0.9330)
Saved: advanced_figures/threshold_opt.png

At threshold=0.595:
  Acc=0.9664
  F1=0.9663
  MCC=0.9329
  FP=16, FN=18


## 10. Comprehensive Confusion Matrices — All Best Models

In [126]:
# ── Confusion matrices for top models ────────────────────────────────────
best_models_results = [
    res_xgb_base,
    res_xgb_opt,
    res_rf_opt,
    res_stack,
    res_vote,
]

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, res in zip(axes, best_models_results):
    cm_norm = res['cm'].astype(float) / res['cm'].sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Ham','Spam'])
    ax.set_yticklabels(['Ham','Spam'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    short = res['name'].replace('XGBoost','XGB').replace('Random Forest','RF')
    ax.set_title(f"{short}\nAcc={res['acc']:.4f}", fontsize=8)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm_norm[i,j]:.3f}\n({res['cm'][i,j]})",
                    ha='center', va='center', fontsize=8,
                    color='white' if cm_norm[i,j] > 0.6 else 'black')
plt.suptitle('Normalised Confusion Matrices — Top Models', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('advanced_figures/all_cm_advanced.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/all_cm_advanced.png')

Saved: advanced_figures/all_cm_advanced.png


## 11. Publication-Ready Results Table

In [127]:
# ── Collect all results into publication table ────────────────────────────
all_results = [
    res_xgb_base,
    res_rf_base,
    res_xgb_opt,
    res_rf_opt,
    res_pos,
    res_top30,
    res_vote,
    res_stack,
]

rows = []
for r in all_results:
    rows.append({
        'Model': r['name'],
        'Accuracy': f"{r['acc']:.4f}",
        'Bal.Acc': f"{r['bal_acc']:.4f}",
        'F1': f"{r['f1']:.4f}",
        'Precision': f"{r['prec']:.4f}",
        'Recall': f"{r['rec']:.4f}",
        'AUC-ROC': f"{r['auc']:.4f}",
        'AP': f"{r['ap']:.4f}",
        'MCC': f"{r['mcc']:.4f}",
        'Kappa': f"{r['kappa']:.4f}",
        'Brier': f"{r['brier']:.4f}",
        'FP': r['fp'],
        'FN': r['fn'],
    })

pub_df = pd.DataFrame(rows)
pub_df.to_csv('advanced_results.csv', index=False)
print('=== Publication Results Table ===')
print(pub_df.to_string(index=False))
print('\nSaved: advanced_results.csv')

=== Publication Results Table ===
                        Model Accuracy Bal.Acc     F1 Precision Recall AUC-ROC     AP    MCC  Kappa  Brier  FP  FN
            XGBoost (default)   0.9595  0.9595 0.9597    0.9550 0.9644  0.9910 0.9887 0.9191 0.9191 0.0294  23  18
                 RF (default)   0.9536  0.9536 0.9534    0.9563 0.9506  0.9877 0.9866 0.9072 0.9072 0.0381  22  25
             XGBoost (Optuna)   0.9625  0.9625 0.9627    0.9570 0.9684  0.9909 0.9845 0.9250 0.9250 0.0292  22  16
                  RF (Optuna)   0.9576  0.9575 0.9573    0.9621 0.9526  0.9880 0.9875 0.9151 0.9151 0.0381  19  24
 XGB-Opt, +perm features (54)   0.9645  0.9645 0.9647    0.9572 0.9723  0.9909 0.9840 0.9290 0.9289 0.0286  22  14
     XGB-Opt, top-30 features   0.9664  0.9664 0.9667    0.9574 0.9763  0.9900 0.9837 0.9331 0.9329 0.0299  22  12
  Soft Voting (XGB+RF+SVM+GB)   0.9605  0.9605 0.9605    0.9605 0.9605  0.9897 0.9834 0.9210 0.9210 0.0320  20  20
Stacking (XGB+RF+SVM+GB → LR)   0.9635  0.9635

In [128]:
# ── Final comparison bar chart ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names   = [r['name'] for r in all_results]
accs    = [r['acc'] for r in all_results]
aucs    = [r['auc'] for r in all_results]
mccs    = [r['mcc'] for r in all_results]

colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(all_results)))
short_names = [
    n.replace('XGBoost','XGB').replace('Random Forest','RF')
     .replace('Stacking','Stack').replace(' (Optuna)','\n(Opt)')
     .replace(' (default)','\n(def)')
    for n in names
]

for ax, vals, metric, lo in zip(axes,
    [accs, aucs, mccs],
    ['Accuracy', 'AUC-ROC', 'MCC'],
    [0.93, 0.985, 0.86]):
    bars = ax.bar(range(len(all_results)), vals, color=colors, edgecolor='white')
    ax.set_ylim(lo, 1.0)
    ax.set_xticks(range(len(all_results)))
    ax.set_xticklabels(short_names, rotation=35, ha='right', fontsize=7)
    ax.set_ylabel(metric); ax.set_title(metric)
    ax.grid(axis='y', alpha=0.3)
    best_i = int(np.argmax(vals))
    bars[best_i].set_edgecolor('red')
    bars[best_i].set_linewidth(2)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=6)

plt.suptitle('Advanced Experiments — Final Model Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('advanced_figures/final_comparison_advanced.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/final_comparison_advanced.png')

Saved: advanced_figures/final_comparison_advanced.png


## 12. CV Score Summary for Paper (10-fold, 5 models)
Formatted table showing mean ± std across 10 folds — ready for the paper's results section.

In [129]:
# ── 10-fold CV + test summary ─────────────────────────────────────────────
print('Computing 10-fold CV for all key models...')
summary_models = {
    'XGB-Default (orig)': (
        XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1),
        X_train_s, X_test_s),
    'XGB-Default (eng)': (
        XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1),
        X_train_es, X_test_es),
    'XGB-Optuna (eng)': (
        XGBClassifier(**xgb_best_params),
        X_train_es, X_test_es),
    'RF-Optuna (eng)': (
        RandomForestClassifier(**rf_best_params),
        X_train_es, X_test_es),
    'Stacking (eng)': (
        StackingClassifier(
            estimators=base_estimators,
            final_estimator=meta_lr,
            cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
            stack_method='predict_proba', n_jobs=-1
        ), X_train_es, X_test_es),
}

summary_rows = []
for model_name, (clf, Xtr, Xte) in summary_models.items():
    sc = cross_val_score(clf, Xtr, y_train, cv=cv10, scoring='accuracy', n_jobs=-1)
    fitted = copy.deepcopy(clf)
    fitted.fit(Xtr, y_train)
    yp    = fitted.predict(Xte)
    yprob = fitted.predict_proba(Xte)[:, 1]
    summary_rows.append({
        'Model': model_name,
        'CV Acc (mean)': sc.mean(),
        'CV Acc (std)':  sc.std(),
        'Test Acc':      accuracy_score(y_test, yp),
        'Test AUC':      roc_auc_score(y_test, yprob),
        'Test F1':       f1_score(y_test, yp),
        'Test MCC':      matthews_corrcoef(y_test, yp),
    })
    print(f"  {model_name:<30} CV={sc.mean():.4f}±{sc.std():.4f}  Test={accuracy_score(y_test,yp):.4f}")

sum_df = pd.DataFrame(summary_rows)
sum_df.to_csv('advanced_summary.csv', index=False)
print('\nSaved: advanced_summary.csv')

# Format for display
sum_df['CV Acc'] = sum_df.apply(
    lambda r: f"{r['CV Acc (mean)']:.4f} ± {r['CV Acc (std)']:.4f}", axis=1)
print('\n' + sum_df[['Model','CV Acc','Test Acc','Test AUC','Test F1','Test MCC']].to_string(index=False))

Computing 10-fold CV for all key models...
  XGB-Default (orig)             CV=0.9570±0.0094  Test=0.9576
  XGB-Default (eng)              CV=0.9553±0.0108  Test=0.9595
  XGB-Optuna (eng)               CV=0.9595±0.0081  Test=0.9625
  RF-Optuna (eng)                CV=0.9467±0.0090  Test=0.9576
  Stacking (eng)                 CV=0.9590±0.0092  Test=0.9635

Saved: advanced_summary.csv

             Model          CV Acc  Test Acc  Test AUC  Test F1  Test MCC
XGB-Default (orig) 0.9570 ± 0.0094    0.9576    0.9909   0.9576    0.9151
 XGB-Default (eng) 0.9553 ± 0.0108    0.9595    0.9910   0.9597    0.9191
  XGB-Optuna (eng) 0.9595 ± 0.0081    0.9625    0.9909   0.9627    0.9250
   RF-Optuna (eng) 0.9467 ± 0.0090    0.9576    0.9880   0.9573    0.9151
    Stacking (eng) 0.9590 ± 0.0092    0.9635    0.9903   0.9634    0.9270


In [130]:
# ── Final summary figure: CV vs Test accuracy with CI bars ────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(sum_df))
w = 0.35
bars1 = ax.bar(x - w/2, sum_df['CV Acc (mean)'],  w,
               yerr=sum_df['CV Acc (std)'] * 1.96,
               capsize=4, label='CV Accuracy (mean ± 1.96σ)', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + w/2, sum_df['Test Acc'], w,
               label='Test Accuracy', color='#FF9800', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(sum_df['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylim(0.93, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('CV Accuracy vs Hold-Out Test Accuracy — All Models')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.001,
            f'{h:.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('advanced_figures/cv_vs_test_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: advanced_figures/cv_vs_test_summary.png')
print('\nAll advanced experiments complete.')

Saved: advanced_figures/cv_vs_test_summary.png

All advanced experiments complete.
